# European Option Pricing Engine

This notebook demonstrates three self-contained methods for pricing European options:

| Method | Approach | Error |
|---|---|---|
| **Black-Scholes** | Closed-form PDE solution | Machine precision |
| **Monte Carlo** | Risk-neutral simulation | $O(1/\sqrt{N})$ stochastic |
| **Binomial tree** | Cox-Ross-Rubinstein lattice | $O(1/N)$ deterministic |

All three methods implemented from scratch using `numpy` and `scipy` only — no pricing libraries.  
Source: [`pricing.py`](pricing.py). Tests: [`tests/test_pricing.py`](tests/test_pricing.py).

---
## 1. Mathematical Background

### The Black-Scholes PDE

Assume the underlying $S_t$ follows geometric Brownian motion under the real-world measure $\mathbb{P}$:

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

Black and Scholes (1973) showed that a self-financing, continuously rebalanced delta-hedge portfolio eliminates all randomness. The resulting no-arbitrage condition on any derivative price $V(S, t)$ is the **Black-Scholes PDE**:

$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0$$

Notice $\mu$ has vanished — the drift of the underlying is irrelevant to the price. This is the content of risk-neutral pricing.

### Risk-Neutral Pricing

By Girsanov's theorem, there exists a measure $\mathbb{Q}$ under which $S_t$ grows at the risk-free rate $r$:

$$dS_t = r S_t \, dt + \sigma S_t \, d\tilde{W}_t \quad \text{under } \mathbb{Q}$$

Any European payoff $h(S_T)$ is priced as its discounted expectation under $\mathbb{Q}$:

$$V(S, t) = e^{-r(T-t)} \, \mathbb{E}^{\mathbb{Q}}\bigl[h(S_T) \,\big|\, \mathcal{F}_t\bigr]$$

### Why a Closed Form Exists for European Options

Under $\mathbb{Q}$, the terminal log-price is Gaussian:

$$\ln S_T = \ln S + \left(r - \tfrac{1}{2}\sigma^2\right)T + \sigma\sqrt{T}\, Z, \quad Z \sim \mathcal{N}(0,1)$$

For a call payoff $\max(S_T - K, 0)$, the expectation $\mathbb{E}^{\mathbb{Q}}[\max(S_T-K,0)]$ reduces to an integral over a truncated log-normal. That integral has an analytic solution in terms of the standard normal CDF $N(\cdot)$:

$$C = S\,N(d_1) - K e^{-rT}\,N(d_2)$$

$$d_1 = \frac{\ln(S/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

- $N(d_2)$: risk-neutral probability that the option expires in-the-money  
- $N(d_1)$: the delta — units of stock held in the replicating portfolio  
- $Ke^{-rT}N(d_2)$: present value of the strike paid at expiry

By put-call parity, $P = C - S + Ke^{-rT}$, so one formula prices both.

---
## 2. Closed-Form Black-Scholes Pricing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from pricing import black_scholes, monte_carlo_price, binomial_tree_price

# Reference option used throughout the notebook
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20

bs_call = black_scholes(S, K, T, r, sigma, "call")
bs_put  = black_scholes(S, K, T, r, sigma, "put")

parity_lhs = bs_call - bs_put
parity_rhs = S - K * np.exp(-r * T)

print(f"Parameters: S={S}, K={K}, T={T}, r={r}, σ={sigma}")
print(f"\nBlack-Scholes call : {bs_call:.6f}")
print(f"Black-Scholes put  : {bs_put:.6f}")
print(f"\nPut-call parity check")
print(f"  C - P          = {parity_lhs:.6f}")
print(f"  S - K·exp(-rT) = {parity_rhs:.6f}")
print(f"  Error          = {abs(parity_lhs - parity_rhs):.2e}")

---
## 3. Monte Carlo Convergence

In [ ]:
N_SIMS_GRID = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
N_TRIALS    = 50

mc_means = np.empty(len(N_SIMS_GRID))
mc_stds  = np.empty(len(N_SIMS_GRID))

rng = np.random.default_rng(0)

for i, n in enumerate(N_SIMS_GRID):
    trial_prices = np.array([
        monte_carlo_price(S, K, T, r, sigma, "call", n, seed=int(rng.integers(1e6)))[0]
        for _ in range(N_TRIALS)
    ])
    mc_means[i] = trial_prices.mean()
    mc_stds[i]  = trial_prices.std(ddof=1)

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 5))

ax.errorbar(
    N_SIMS_GRID, mc_means, yerr=mc_stds,
    fmt="o-", color="steelblue", ecolor="steelblue", elinewidth=1.2,
    capsize=4, linewidth=1.5, markersize=5, label="MC price (mean ± std over 50 trials)"
)
ax.axhline(bs_call, color="firebrick", linewidth=1.5, linestyle="--", label=f"Black-Scholes = {bs_call:.4f}")

ax.set_xscale("log")
ax.set_xlabel("Number of simulations (log scale)", fontsize=12)
ax.set_ylabel("Call price", fontsize=12)
ax.set_title("Monte Carlo convergence to Black-Scholes", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, which="both", linestyle=":", linewidth=0.6, alpha=0.7)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
fig.tight_layout()
plt.show()

---
## 4. Binomial Tree Convergence

In [ ]:
N_STEPS_GRID = [10, 25, 50, 100, 250, 500, 1_000, 2_500]

bt_prices = np.array([
    binomial_tree_price(S, K, T, r, sigma, "call", n)
    for n in N_STEPS_GRID
])

errors = bt_prices - bs_call

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: absolute price vs BS
ax = axes[0]
ax.plot(N_STEPS_GRID, bt_prices, "o-", color="steelblue", linewidth=1.5, markersize=5,
        label="Binomial tree price")
ax.axhline(bs_call, color="firebrick", linewidth=1.5, linestyle="--",
           label=f"Black-Scholes = {bs_call:.4f}")
ax.set_xscale("log")
ax.set_xlabel("Number of steps (log scale)", fontsize=12)
ax.set_ylabel("Call price", fontsize=12)
ax.set_title("CRR binomial tree: price vs. steps", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, which="both", linestyle=":", linewidth=0.6, alpha=0.7)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Right: signed error — reveals oscillation pattern
ax = axes[1]
ax.plot(N_STEPS_GRID, errors, "o-", color="darkorange", linewidth=1.5, markersize=5,
        label="BT price − BS price")
ax.axhline(0, color="firebrick", linewidth=1.2, linestyle="--", label="Zero error")
ax.set_xscale("log")
ax.set_xlabel("Number of steps (log scale)", fontsize=12)
ax.set_ylabel("Pricing error (BT − BS)", fontsize=12)
ax.set_title("CRR binomial tree: signed error (oscillation)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, which="both", linestyle=":", linewidth=0.6, alpha=0.7)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.tight_layout()
plt.show()

---
## 5. Findings

### Monte Carlo: stochastic error shrinks as $1/\sqrt{N}$

The error bars in the MC convergence plot contract predictably as simulation count grows. This is a direct consequence of the Central Limit Theorem: the standard error of the sample mean of $N$ independent payoffs is $\hat{\sigma}/\sqrt{N}$, where $\hat{\sigma}$ is the cross-simulation standard deviation of discounted payoffs. Doubling precision requires quadrupling simulations — the fundamental cost of MC. At 500,000 simulations the mean price sits within a few cents of the Black-Scholes value, with trial-to-trial spread below $0.05. The method is unbiased in the limit but never exact for any finite $N$.

### Binomial tree: deterministic convergence with odd/even oscillation

The right-hand panel makes the oscillation pattern explicit. The CRR tree alternately over- and under-prices the option as $N$ increases, producing a signed error that flips sign at each step and decays in amplitude at rate $O(1/N)$. The source is the parity mismatch between the strike $K$ and the discrete lattice nodes: when $N$ is even versus odd, the moneyness of the nearest node straddles the strike differently, introducing a systematic bias that alternates. This is not numerical instability — it is a structural artefact of the discrete lattice. Practitioners who use binomial trees for Greeks often average results from $N$ and $N+1$ steps to cancel the oscillation.

### When to use each method in practice

Black-Scholes closed form is the only choice when speed matters (real-time Greeks, large options books) and model assumptions hold. Monte Carlo is indispensable when the payoff is path-dependent (Asian, barrier, lookback) or the state space has many dimensions — it scales in cost independent of dimensionality, unlike trees. The binomial tree is the natural tool for American option pricing, where early-exercise decisions are made at every node, and for building intuition about hedging because the replicating portfolio weights are visible at each step. For European vanilla options, as benchmarked here, Black-Scholes dominates on both speed and accuracy.

---
## 6. Limitations

**European exercise only.** All three methods price contracts that can only be exercised at maturity $T$. American options, which permit early exercise at any $t \leq T$, require dynamic programming (binomial tree with early-exercise check, or least-squares Monte Carlo à la Longstaff-Schwartz). The Black-Scholes formula has no analytic extension to American options on dividend-paying stocks.

**Constant volatility.** Black-Scholes assumes $\sigma$ is a fixed parameter. In practice, implied volatility varies with both strike (the *volatility smile*) and maturity (the *term structure*), forming the volatility surface. A single $\sigma$ cannot simultaneously match market prices across strikes — this is the primary reason practitioners use local volatility (Dupire 1994) or stochastic volatility models (Heston 1993) in production.

**No dividends, no transaction costs, continuous trading.** The model assumes the underlying pays no dividends (trivially extended via a dividend yield $q$ in the drift term, but not implemented here), that the hedge can be rebalanced continuously without cost, and that short-selling is unrestricted. Transaction costs in a delta-hedging strategy accumulate at a rate proportional to $\Gamma$ and the bid-ask spread — their practical impact is entirely absent from this framework.